In [1]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / ".git").exists():
            return p
    raise RuntimeError("Could not locate project root (no .git found above cwd)")

PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"


# Online Batch Learning - Data Preparation
Prepare data for online/incremental learning models using scikit-learn

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from tqdm.notebook import tqdm
from src.cube.prepare import restore_dataset_attrs
import warnings
# warnings.filterwarnings('ignore')

## 1. Load Data from Zarr

In [3]:
OPEN_FOLDER = PROJECT_ROOT
DATA_PATH = OPEN_FOLDER / 'full_dataset_resizedv2.zarr'

print('Loading data from zarr...')
data = xr.open_dataset(DATA_PATH, engine='zarr', mask_and_scale=True)
data = restore_dataset_attrs(data)
print('✓ Data loaded successfully')
print(f'Data shape: {data}')

Loading data from zarr...


✓ Data loaded successfully


Data shape: <xarray.Dataset> Size: 67GB
Dimensions:               (cube: 2216, year: 7, topK: 4, y: 128, x: 128,
                           s2_band: 7, norm: 4, bin_dim: 1, scl_class: 12,
                           y_300: 9, x_300: 9, pos: 2, time: 495,
                           weather_band: 27, stat_weather: 162, statistic: 6)
Coordinates: (12/15)
  * cube                  (cube) <U31 275kB 'mc_-0.04_51.09_1.2_20230627_0' ....
  * year                  (year) int64 56B 2016 2017 2018 2019 2020 2021 2022
  * topK                  (topK) int64 32B 0 1 2 3
  * y                     (y) int64 1kB 127 126 125 124 123 122 ... 5 4 3 2 1 0
  * x                     (x) int64 1kB 0 1 2 3 4 5 ... 122 123 124 125 126 127
  * s2_band               (s2_band) <U3 84B 'B02' 'B03' 'B04' ... 'B07' 'B8A'
    ...                    ...
  * x_300                 (x_300) int64 72B 0 1 2 3 4 5 6 7 8
  * pos                   (pos) <U3 24B 'lat' 'lon'
  * time                  (time) datetime64[ns] 4kB 20

In [4]:
# Show dataset variables and dimensions
rows = []
for name, var in data.data_vars.items():
    rows.append({
        'name': name,
        'dims': tuple(var.dims),
        'dtype': str(var.dtype)
    })
var_overview = pd.DataFrame(rows)
display(var_overview)
print(f'Total variables: {len(var_overview)}')

,name,dims,dtype
0,S2,"(cube, year, topK, y, x, s2_band)",float64
1,S2_norm,"(norm, s2_band)",float32
2,SCL,"(cube, year, topK, y, x, bin_dim)",uint8
3,SCL_count,"(cube, year, y, x, scl_class)",uint16
4,SCL_count_norm,"(norm, scl_class)",float32
5,anyhist_disturbances,"(cube, year, y, x)",uint8
6,cloudmask,"(cube, year, topK, y, x, bin_dim)",uint8
7,dem,"(cube, y, x, bin_dim)",float64
8,dem_norm,"(norm, bin_dim)",float32
9,disturbance_agent,"(cube, year, y, x)",uint8


Total variables: 26


## 2. Filter Cubes with Disturbances

In [5]:
# Keep only cubes with at least one disturbance in any year
print('Filtering cubes with disturbances...')
original_cube_count = data.sizes["cube"]
print(f'Original cubes: {original_cube_count}')

# Determine spatial dimensions
if 'x_30' in data.dims and 'y_30' in data.dims:
    spatial_dims = ['year', 'x_30', 'y_30']
elif 'x' in data.dims and 'y' in data.dims:
    spatial_dims = ['year', 'x', 'y']
else:
    spatial_dims = ['year']

# Mask: True if cube has at least one disturbance pixel in any year
has_disturbance = (data['disturbances'] == 1).any(dim=spatial_dims)

# Filter to keep only cubes with disturbances
cubes_with_disturbance = data['cube'].where(has_disturbance, drop=True).values
data = data.isel(cube=np.isin(data['cube'].values, cubes_with_disturbance))

cubes_after_filter = data.sizes["cube"]
cubes_filtered_out = original_cube_count - cubes_after_filter

print(f'Cubes with disturbances: {cubes_after_filter}')
print(f'Cubes filtered out: {cubes_filtered_out}')

Filtering cubes with disturbances...
Original cubes: 2216


Cubes with disturbances: 1975
Cubes filtered out: 241


## 3. Pixel-wise Sampling Strategy
Extract pixels using the same strategy as XGBoost notebook:
- Find all disturbed pixels
- Add 6x6 neighborhood around each disturbed pixel
- Add 5 random pixels per disturbed pixel (for balance)
- Extract features across all years

In [6]:
# Configuration
TEST_MODE = False  # Set to True to test on a single cube
TEST_CUBE_INDEX = 0  # Used only when TEST_MODE is True

# Sampler configuration. Every setting that changes which pixels are sampled, or the order
# they end up in, lives here so the manifest written alongside training_data.zarr records
# the exact run that produced it. Downstream artifacts (data_split.npz, the disk feature
# cache, everything under experiments/) address pixels positionally, so an unrecorded
# change here silently invalidates all of them. See src/cube/manifest.py.
RANDOM_SEED = 42
NEIGHBOURHOOD_RADIUS = 3  # 7x7 window centred on each disturbed pixel
NEGATIVES_PER_DISTURBED_PIXEL = 5
S2_TOPK_INDEX = 0

rng = np.random.default_rng(RANDOM_SEED)

print('Starting pixel-wise sampling...')
print(f'  seed={RANDOM_SEED}, neighbourhood_radius={NEIGHBOURHOOD_RADIUS}, '
      f'negatives_per_disturbed_pixel={NEGATIVES_PER_DISTURBED_PIXEL}, s2_topK={S2_TOPK_INDEX}')

# Get spatial dimensions
y_dim = 'y'
x_dim = 'x'
y_size = data.sizes[y_dim]
x_size = data.sizes[x_dim]
n_years = data.sizes['year']
n_s2_bands = data.sizes['s2_band']

print(f'Grid size: {y_size} x {x_size}, Years: {n_years}, S2 bands: {n_s2_bands}')

# Lists to accumulate pixel data
cube_indices = []
cube_names = []
x_coords = []
y_coords = []
disturbances_list = []
s2_list = []
dem_list = []
year_disturbance_list = []  # Track year of disturbance for each pixel

# Configure which cubes to iterate
if TEST_MODE:
    cube_iter = [int(TEST_CUBE_INDEX)]
else:
    cube_iter = range(data.sizes['cube'])

for cube_idx in tqdm(cube_iter, total=len(cube_iter), desc='Processing cubes'):
    # Load all needed data for this cube at once
    dist_cube = data['disturbances'].isel(cube=cube_idx).values  # (year, y, x)
    s2_cube = data['S2'].isel(cube=cube_idx, topK=S2_TOPK_INDEX).values  # (year, y, x, s2_band)
    dem_cube = data['dem'].isel(cube=cube_idx).values  # (y, x, bin_dim=1)
    dem_cube = dem_cube[..., 0]  # squeeze bin_dim -> (y, x)
    forest_mask_cube = data['forest_mask'].isel(cube=cube_idx).values  # (year, y, x) or (y, x)
    cube_name = str(data['cube'].isel(cube=cube_idx).values)

    # Create forest mask across all years
    if forest_mask_cube.ndim == 3:  # (year, y, x)
        forest_mask_any_year = (forest_mask_cube == 1).any(axis=0)  # (y, x)
    else:  # (y, x)
        forest_mask_any_year = (forest_mask_cube == 1)  # (y, x)

    # Find all pixels that have disturbance in ANY year AND are in forest
    disturbed_mask = (dist_cube == 1).any(axis=0)  # (y, x)
    disturbed_mask = disturbed_mask & forest_mask_any_year  # Apply forest mask filter
    disturbed_pixels = np.argwhere(disturbed_mask)  # (N, 2) list of (y, x)

    # Set to track sampled pixels (no duplicates)
    sampled_pixels = set()

    # Step 1: Add disturbed pixels and their neighborhoods (7x7 window centered on pixel)
    for yp, xp in disturbed_pixels:
        # Add the disturbed pixel itself
        sampled_pixels.add((int(yp), int(xp)))
        # Add neighborhood
        y0 = max(0, yp - NEIGHBOURHOOD_RADIUS)
        y1 = min(y_size, yp + NEIGHBOURHOOD_RADIUS + 1)
        x0 = max(0, xp - NEIGHBOURHOOD_RADIUS)
        x1 = min(x_size, xp + NEIGHBOURHOOD_RADIUS + 1)
        for ny in range(y0, y1):
            for nx in range(x0, x1):
                sampled_pixels.add((int(ny), int(nx)))

    # Step 2: Add N random pixels for EACH disturbed pixel (unique overall)
    if len(disturbed_pixels) > 0:
        taken_mask = np.zeros((y_size, x_size), dtype=bool)
        for (ty, tx) in sampled_pixels:
            taken_mask[ty, tx] = True
        # Only sample from forest pixels not already taken
        available_mask = (~taken_mask) & forest_mask_any_year
        available_idx = np.flatnonzero(available_mask.ravel())
        n_to_pick = min(NEGATIVES_PER_DISTURBED_PIXEL * len(disturbed_pixels), available_idx.size)
        if n_to_pick > 0:
            # Seeded generator, advanced once per cube: reproducible given RANDOM_SEED and
            # the cube iteration order, both of which the manifest records.
            picked = rng.choice(available_idx, n_to_pick, replace=False)
            ys = picked // x_size
            xs = picked % x_size
            for ry, rx in zip(ys, xs):
                sampled_pixels.add((int(ry), int(rx)))

    # Step 3: Extract features for all sampled pixels
    # Sorted so pixel order is a deterministic function of which pixels were sampled, rather
    # than of Python set iteration order. data_split.npz indexes this dimension positionally,
    # so the ordering is part of the dataset's identity, not an implementation detail.
    for yp, xp in sorted(sampled_pixels):
        cube_indices.append(int(cube_idx))
        cube_names.append(cube_name)
        x_coords.append(int(xp))
        y_coords.append(int(yp))

        # Disturbances: (year,) vector
        disturbances_list.append(dist_cube[:, yp, xp])

        # S2 bands: (year, s2_band) (topK already selected)
        s2_list.append(s2_cube[:, yp, xp, :])

        # DEM: scalar per pixel, replicate across years
        dem_value = dem_cube[yp, xp]
        dem_pixel_repeated = np.full((n_years,), dem_value)
        dem_list.append(dem_pixel_repeated)

        # Year of first disturbance (0 if never disturbed)
        dist_years = np.where(dist_cube[:, yp, xp] == 1)[0]
        year_disturbance_list.append(dist_years[0] if len(dist_years) > 0 else -1)

print(f'\nTotal sampled pixels across all cubes: {len(cube_indices)}')

Starting pixel-wise sampling...
  seed=42, neighbourhood_radius=3, negatives_per_disturbed_pixel=5, s2_topK=0
Grid size: 128 x 128, Years: 7, S2 bands: 7


Processing cubes:   0%|          | 0/1975 [00:00<?, ?it/s]


Total sampled pixels across all cubes: 8155205


## 4. Convert to Structured Format

In [7]:





# Convert to arrays
print('Converting sampled data to arrays...')

n_pixels = len(cube_indices)
print(f'Total pixels: {n_pixels}')

cube_indices_arr = np.array(cube_indices)
cube_names_arr = np.array(cube_names)
x_coords_arr = np.array(x_coords)
y_coords_arr = np.array(y_coords)
disturbances_arr = np.stack(disturbances_list)  # (n_pixels, n_years)
s2_arr = np.stack(s2_list)  # (n_pixels, n_years, n_s2_bands)
dem_arr = np.stack(dem_list)  # (n_pixels, n_years)
year_disturbance_arr = np.array(year_disturbance_list)  # (n_pixels,)

print(f'\nArray shapes:')
print(f'  cube_indices: {cube_indices_arr.shape}')
print(f'  disturbances: {disturbances_arr.shape}')
print(f'  s2_bands: {s2_arr.shape}')
print(f'  dem: {dem_arr.shape}')
print(f'  year_disturbance: {year_disturbance_arr.shape}')

Converting sampled data to arrays...
Total pixels: 8155205



Array shapes:
  cube_indices: (8155205,)
  disturbances: (8155205, 7)
  s2_bands: (8155205, 7, 7)
  dem: (8155205, 7)
  year_disturbance: (8155205,)


In [8]:
# Create xarray dataset for easy manipulation
pixel_dim = np.arange(n_pixels)

training_dataset = xr.Dataset(
    data_vars={
        'cube_idx': (['pixel'], cube_indices_arr),
        'cube_name': (['pixel'], cube_names_arr),
        'x': (['pixel'], x_coords_arr),
        'y': (['pixel'], y_coords_arr),
        'disturbances': (['pixel', 'year'], disturbances_arr),
        's2_bands': (['pixel', 'year', 's2_band'], s2_arr),
        'dem': (['pixel', 'year'], dem_arr),
        'year_disturbance': (['pixel'], year_disturbance_arr),
    },
    coords={
        'pixel': pixel_dim,
        'year': data['year'].values,
        's2_band': data['s2_band'].values,
    }
)

print('Training dataset created:')
print(training_dataset)

Training dataset created:
<xarray.Dataset> Size: 5GB
Dimensions:           (pixel: 8155205, year: 7, s2_band: 7)
Coordinates:
  * pixel             (pixel) int32 33MB 0 1 2 3 ... 8155202 8155203 8155204
  * year              (year) int64 56B 2016 2017 2018 2019 2020 2021 2022
  * s2_band           (s2_band) <U3 84B 'B02' 'B03' 'B04' ... 'B06' 'B07' 'B8A'
Data variables:
    cube_idx          (pixel) int32 33MB 0 0 0 0 0 ... 1974 1974 1974 1974 1974
    cube_name         (pixel) <U31 1GB 'mc_-0.04_51.09_1.2_20230627_0' ... 'm...
    x                 (pixel) int32 33MB 28 30 33 34 37 ... 94 102 107 111 127
    y                 (pixel) int32 33MB 0 0 0 0 0 0 ... 127 127 127 127 127 127
    disturbances      (pixel, year) uint8 57MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0
    s2_bands          (pixel, year, s2_band) float64 3GB nan nan ... 0.2838
    dem               (pixel, year) float64 457MB 115.5 115.5 ... 641.0 641.0
    year_disturbance  (pixel) int64 65MB -1 -1 -1 -1 -1 -1 ... -1 -1 -1

In [9]:
def per_year_quality_report(ds, core_vars, label_var='disturbances'):
    """Per-year label distribution, NaN rate, and row-survival for a set of core feature vars.

    A dataset-wide NaN count (as used below) can hide an entire missing year inside a small
    overall percentage -- e.g. 100% NaN in one year and ~0.1% in the rest averages to ~14%
    and reads as unremarkable. Breaking it out by year is what would have caught that.
    `row_survival_pct` mirrors what a feature-prep consumer keeps for that year: a valid
    label (0/1) and no NaN across every var in `core_vars`.

    Computed one year at a time rather than materialising every var's full (pixel, year, ...)
    array up front: s2_bands alone is ~3.2GB as a single .values call on the real dataset, and
    that was enough to exhaust memory on a 15GB machine once stacked on everything else a long
    notebook run had already accumulated by this point. Slicing per year bounds the extra
    working set to one year's worth of one var at a time.
    """
    n_years = ds.sizes['year']
    n_pixels = ds.sizes['pixel']

    rows = []
    for t in range(n_years):
        year_labels = ds[label_var].isel(year=t).values
        n_pos = int(np.sum(year_labels == 1))
        n_neg = int(np.sum(year_labels == 0))
        n_nodata = n_pixels - n_pos - n_neg

        row = {
            'year': int(ds['year'].values[t]),
            'label_pos': n_pos,
            'label_neg': n_neg,
            'label_nodata': n_nodata,
            'pos_rate_pct': round(100.0 * n_pos / (n_pos + n_neg), 2) if (n_pos + n_neg) else float('nan'),
        }

        survive = np.isin(year_labels, [0, 1])
        for var_name in core_vars:
            values = ds[var_name].isel(year=t).values  # (pixel,) or (pixel, band)
            var_nan = np.isnan(values).any(axis=-1) if values.ndim == 2 else np.isnan(values)
            row[f'{var_name}_nan_pct'] = round(100.0 * float(var_nan.sum()) / n_pixels, 2)
            survive = survive & ~var_nan

        row['row_survival_pct'] = round(100.0 * float(survive.sum()) / n_pixels, 2)
        rows.append(row)
    return pd.DataFrame(rows)

In [10]:
# Save to zarr for fast retrieval
from src.cube.manifest import manifest_path_for, write_manifest

output_path = PROJECT_ROOT / 'training_data.zarr'

print(f'Saving to {output_path}...')
training_dataset.to_zarr(output_path, mode='w')
print('✓ Saved successfully!')

# Record what produced this pixel set. The store itself carries no trace of the seed,
# the sampling radius, or the ordering rule, yet every downstream artifact that indexes
# `pixel` positionally depends on all three. The manifest makes a re-run detectable
# instead of silent.
manifest = write_manifest(
    output_path,
    cube_idx=cube_indices_arr,
    y=y_coords_arr,
    x=x_coords_arr,
    cube_names=cube_names_arr,
    source_dataset=DATA_PATH.name,
    config={
        'random_seed': RANDOM_SEED,
        'neighbourhood_radius': NEIGHBOURHOOD_RADIUS,
        'negatives_per_disturbed_pixel': NEGATIVES_PER_DISTURBED_PIXEL,
        's2_topk_index': S2_TOPK_INDEX,
        'test_mode': TEST_MODE,
        'cube_filter': 'keep cubes with at least one disturbance pixel in any year',
        'pixel_filter': (
            'forest pixels disturbed in any year, plus their neighbourhood window, plus '
            'random not-yet-sampled forest pixels'
        ),
        'pixel_order': 'cube index ascending, then (y, x) ascending within each cube',
    },
    extra={
        'n_cubes_before_filter': int(original_cube_count),
        'n_cubes_after_filter': int(cubes_after_filter),
        'n_pixels': int(n_pixels),
        'n_years': int(n_years),
        'n_s2_bands': int(n_s2_bands),
    },
)

print(f'✓ Manifest written to {manifest_path_for(output_path).name}')
print(f"  pixel_identity_sha256: {manifest['pixel_identity_sha256']}")

Saving to C:\Users\bartu\Desktop\Fonda-scikit - Git\training_data.zarr...


C:\Users\bartu\Desktop\Fonda-scikit - Git\venv\Lib\site-packages\zarr\core\dtype\npy\string.py:248: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=3, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)


C:\Users\bartu\Desktop\Fonda-scikit - Git\venv\Lib\site-packages\zarr\core\dtype\npy\string.py:248: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=31, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)


C:\Users\bartu\Desktop\Fonda-scikit - Git\venv\Lib\site-packages\zarr\api\asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


✓ Saved successfully!


✓ Manifest written to training_data.manifest.json
  pixel_identity_sha256: 38c8d32300cda396519c585c12120494fe85424e90135de6c3eec11f30f9eeda


In [11]:
# Reload and verify saved dataset
ds_path = PROJECT_ROOT / 'training_data.zarr'
print(f'Loading saved dataset from {ds_path}...')
training_dataset = xr.open_dataset(ds_path, engine='zarr')
print('✓ Dataset loaded successfully\n')

# Data quality checks and statistics
print('=== DATASET QUALITY CHECKS ===\n')

# 1. Dataset overview
print('Dataset Overview:')
print(f'  Total pixels: {len(training_dataset.pixel)}')
print(f'  Total years: {len(training_dataset.year)}')
print(f'  Total observations: {len(training_dataset.pixel) * len(training_dataset.year)}')

# 2. Disturbance statistics
disturbances_flat = training_dataset['disturbances'].values.flatten()
n_disturbed = int(np.sum(disturbances_flat == 1))
n_not_disturbed = int(np.sum(disturbances_flat == 0))
total_labels = n_disturbed + n_not_disturbed

print(f'\nDisturbance Label Distribution:')
print(f'  Disturbed (1): {n_disturbed:,} ({100.0*n_disturbed/total_labels:.1f}%)')
print(f'  Not disturbed (0): {n_not_disturbed:,} ({100.0*n_not_disturbed/total_labels:.1f}%)')

# 3. Year disturbance statistics
year_dist = training_dataset['year_disturbance'].values
n_never_disturbed = np.sum(year_dist == -1)
n_with_disturbance = np.sum(year_dist >= 0)

print(f'\nYear of First Disturbance:')
print(f'  Never disturbed (-1): {n_never_disturbed}')
print(f'  Has disturbance (>=0): {n_with_disturbance}')
if n_with_disturbance > 0:
    valid_years = year_dist[year_dist >= 0]
    print(f'  Year range: {int(np.min(valid_years))} to {int(np.max(valid_years))}')
    print(f'  Mean year: {np.mean(valid_years):.1f}')

# 4. Feature array shapes
print(f'\nFeature Array Shapes:')
print(f'  cube_idx: {training_dataset["cube_idx"].shape}')
print(f'  cube_name: {training_dataset["cube_name"].shape}')
print(f'  x: {training_dataset["x"].shape}')
print(f'  y: {training_dataset["y"].shape}')
print(f'  disturbances: {training_dataset["disturbances"].shape}')
print(f'  s2_bands: {training_dataset["s2_bands"].shape}')
print(f'  dem: {training_dataset["dem"].shape}')

# 5. Data type checks
print(f'\nData Type Verification:')
print(f'  cube_idx: {training_dataset["cube_idx"].dtype}')
print(f'  disturbances: {training_dataset["disturbances"].dtype}')
print(f'  s2_bands: {training_dataset["s2_bands"].dtype}')
print(f'  dem: {training_dataset["dem"].dtype}')

# 6. NaN/Missing value checks (dataset-wide -- see the per-year table below for why this
# alone is not enough to trust)
print(f'\nMissing Value Checks:')
print(f'  disturbances NaNs: {np.isnan(training_dataset["disturbances"].values).sum()}')
print(f'  s2_bands NaNs: {np.isnan(training_dataset["s2_bands"].values).sum()}')
print(f'  dem NaNs: {np.isnan(training_dataset["dem"].values).sum()}')

# 7. Per-year quality report -- a dataset-wide NaN count can hide an entire missing year
print(f'\nPer-Year Quality Report:')
display(per_year_quality_report(training_dataset, core_vars=['s2_bands', 'dem']))

# 8. Sample data
print(f'\nSample of first 3 pixels:')
for i in range(min(3, len(training_dataset.pixel))):
    pixel_data = training_dataset.isel(pixel=i)
    print(f'  Pixel {i}: cube={pixel_data["cube_name"].values}, x={pixel_data["x"].values}, y={pixel_data["y"].values}, '
          f'disturbed_years={np.sum(pixel_data["disturbances"].values == 1)}, first_disturbance_year={pixel_data["year_disturbance"].values}')

# 9. Memory usage
print(f'\nStorage Information:')
print(f'  Memory usage: {training_dataset.nbytes / 1e6:.2f} MB')

Loading saved dataset from C:\Users\bartu\Desktop\Fonda-scikit - Git\training_data.zarr...


✓ Dataset loaded successfully

=== DATASET QUALITY CHECKS ===

Dataset Overview:
  Total pixels: 8155205
  Total years: 7
  Total observations: 57086435



Disturbance Label Distribution:
  Disturbed (1): 1,132,683 (2.0%)
  Not disturbed (0): 55,945,933 (98.0%)

Year of First Disturbance:
  Never disturbed (-1): 7060710
  Has disturbance (>=0): 1094495
  Year range: 0 to 6
  Mean year: 3.2

Feature Array Shapes:
  cube_idx: (8155205,)
  cube_name: (8155205,)
  x: (8155205,)
  y: (8155205,)
  disturbances: (8155205, 7)
  s2_bands: (8155205, 7, 7)
  dem: (8155205, 7)

Data Type Verification:
  cube_idx: int32
  disturbances: uint8
  s2_bands: float64
  dem: float64

Missing Value Checks:


  disturbances NaNs: 0


  s2_bands NaNs: 57434909


  dem NaNs: 19257

Per-Year Quality Report:


,year,label_pos,label_neg,label_nodata,pos_rate_pct,s2_bands_nan_pct,dem_nan_pct,row_survival_pct
0,2016,115361,8038727,1117,1.41,100.00,0.03,0.00
1,2017,150189,8003899,1117,1.84,0.12,0.03,99.84
2,2018,171721,7982367,1117,2.11,0.11,0.03,99.84
3,2019,163167,7990921,1117,2.00,0.12,0.03,99.84
4,2020,157064,7997024,1117,1.93,0.10,0.03,99.85
5,2021,126634,8027454,1117,1.55,0.11,0.03,99.84
6,2022,248547,7905541,1117,3.05,0.05,0.03,99.91



Sample of first 3 pixels:
  Pixel 0: cube=mc_-0.04_51.09_1.2_20230627_0, x=28, y=0, disturbed_years=0, first_disturbance_year=-1
  Pixel 1: cube=mc_-0.04_51.09_1.2_20230627_0, x=30, y=0, disturbed_years=0, first_disturbance_year=-1
  Pixel 2: cube=mc_-0.04_51.09_1.2_20230627_0, x=33, y=0, disturbed_years=0, first_disturbance_year=-1

Storage Information:
  Memory usage: 4917.59 MB


## 5. Compute Spectral Indices

In [12]:
# Load dataset and compute spectral indices with chunking
ds_path = PROJECT_ROOT / 'training_data.zarr'
print('Loading dataset with chunking...')
ds = xr.open_dataset(ds_path, engine='zarr', chunks={'pixel': 10000})
print(f'Dataset loaded with chunks: {ds.chunks}')

print('\nComputing spectral indices...')

band_names = set(ds['s2_band'].values.tolist())
required_bands = {'B03', 'B04', 'B8A'}
missing_bands = required_bands - band_names

if missing_bands:
    raise ValueError(f'Missing required bands: {missing_bands}')

s2 = ds['s2_bands']  # dims: pixel, year, s2_band

# Extract bands
nir = s2.sel(s2_band='B8A')
red = s2.sel(s2_band='B04')
green = s2.sel(s2_band='B03')

# Safe division
def safe_divide(numerator, denominator):
    return xr.where(denominator != 0, numerator / denominator, np.nan)

# NDVI: (NIR - Red) / (NIR + Red)
ndvi = safe_divide(nir - red, nir + red)

# NDWI: (Green - NIR) / (Green + NIR)
ndwi = safe_divide(green - nir, green + nir)

# NBR using B07 as SWIR proxy (if available)
if 'B07' in band_names:
    swir_proxy = s2.sel(s2_band='B07')
    nbr = safe_divide(nir - swir_proxy, nir + swir_proxy)
    ds = ds.assign(ndvi=ndvi, ndwi=ndwi, nbr=nbr)
    print('✓ Added indices: NDVI, NDWI, NBR (B07 as SWIR proxy)')
else:
    ds = ds.assign(ndvi=ndvi, ndwi=ndwi)
    print('✓ Added indices: NDVI, NDWI (B07/SWIR bands unavailable for NBR)')

Loading dataset with chunking...


Dataset loaded with chunks: Frozen({'pixel': (10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10000, 10

C:\Users\bartu\AppData\Local\Temp\ipykernel_1116\3086422200.py:4: UserWarning: The specified chunks separate the stored chunks along dimension "pixel" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(ds_path, engine='zarr', chunks={'pixel': 10000})


In [13]:
import shutil
import time
import tempfile
import dask
import xarray as xr
from pathlib import Path

# 1. Setup paths
output_path = PROJECT_ROOT / 'training_data_enriched.zarr'

print(f'Saving to workspace directory: {output_path.absolute()}')

# 2. Cleanup existing data (handle Windows file locks during deletion)
if output_path.exists():
    print(f'Removing existing {output_path}...')
    try:
        shutil.rmtree(output_path)
        time.sleep(0.5)  # Give Windows a moment to release handles
    except Exception as e:
        print(f'Warning: Could not remove directory: {e}')

# 3. Pre-process strings to avoid "UnstableSpecificationWarning"
# This converts UTF32 strings to a format Zarr handles more gracefully
for var in ds.variables:
    if ds[var].dtype.kind in {'U', 'S'}:  # If it's a Unicode or Byte string
        ds[var] = ds[var].astype(object)

# 4. Apply chunking
# Adjust 'pixel' chunk size based on your RAM if needed
ds_chunked = ds.chunk({'pixel': 100000, 'year': -1})

print(f'Saving enriched dataset to {output_path}...')

# 5. The Critical Windows Fix: Use the 'synchronous' scheduler.
# This avoids 'Access is denied' by ensuring only one file is renamed at a time.
#
# Both except branches re-raise after printing: the next cell reloads this same path and
# reports "Dataset loaded successfully" on whatever it finds there, so swallowing a failed
# write here would let that reload silently succeed against a missing or stale store.
try:
    with dask.config.set(scheduler='synchronous'):
        # Use encoding={} to let xarray create new chunk sizes instead of reusing old ones
        ds_chunked.to_zarr(
            output_path,
            mode='w',
            compute=True,
            encoding={var: {'chunks': None} for var in ds_chunked.data_vars}
        )
    print('✓ Saved successfully!')
    print(f'File location: {output_path}')
except PermissionError:
    print("\n❌ PermissionError still occurring.")
    print("Check if a File Explorer window is open at the destination,")
    print("or if an Antivirus/OneDrive is syncing that folder.")
    raise
except Exception as e:
    print(f"\n❌ An unexpected error occurred: {e}")
    raise

# 6. Verify variables
print(f'\nEnriched dataset variables:')
print(list(ds.data_vars))

Saving to workspace directory: C:\Users\bartu\Desktop\Fonda-scikit - Git\training_data_enriched.zarr
Removing existing C:\Users\bartu\Desktop\Fonda-scikit - Git\training_data_enriched.zarr...


Saving enriched dataset to C:\Users\bartu\Desktop\Fonda-scikit - Git\training_data_enriched.zarr...


C:\Users\bartu\AppData\Local\Temp\ipykernel_1116\2039747124.py:43: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds_chunked.to_zarr(


C:\Users\bartu\Desktop\Fonda-scikit - Git\venv\Lib\site-packages\zarr\api\asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


C:\Users\bartu\Desktop\Fonda-scikit - Git\venv\Lib\site-packages\dask\_task_spec.py:759: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


✓ Saved successfully!
File location: C:\Users\bartu\Desktop\Fonda-scikit - Git\training_data_enriched.zarr

Enriched dataset variables:
['cube_idx', 'cube_name', 'dem', 'disturbances', 's2_bands', 'x', 'y', 'year_disturbance', 'ndvi', 'ndwi', 'nbr']


In [14]:
# Reload and verify saved dataset
ds_path = PROJECT_ROOT / 'training_data_enriched.zarr'
print(f'Loading saved dataset from {ds_path}...')
training_dataset = xr.open_dataset(ds_path, engine='zarr')
print('✓ Dataset loaded successfully\n')

# Data quality checks and statistics
print('=== DATASET QUALITY CHECKS ===\n')

# 1. Dataset overview
print('Dataset Overview:')
print(f'  Total pixels: {len(training_dataset.pixel)}')
print(f'  Total years: {len(training_dataset.year)}')
print(f'  Total observations: {len(training_dataset.pixel) * len(training_dataset.year)}')

# 2. Disturbance statistics
disturbances_flat = training_dataset['disturbances'].values.flatten()
n_disturbed = int(np.sum(disturbances_flat == 1))
n_not_disturbed = int(np.sum(disturbances_flat == 0))
total_labels = n_disturbed + n_not_disturbed

print(f'\nDisturbance Label Distribution:')
print(f'  Disturbed (1): {n_disturbed:,} ({100.0*n_disturbed/total_labels:.1f}%)')
print(f'  Not disturbed (0): {n_not_disturbed:,} ({100.0*n_not_disturbed/total_labels:.1f}%)')

# 3. Year disturbance statistics
year_dist = training_dataset['year_disturbance'].values
n_never_disturbed = np.sum(year_dist == -1)
n_with_disturbance = np.sum(year_dist >= 0)

print(f'\nYear of First Disturbance:')
print(f'  Never disturbed (-1): {n_never_disturbed}')
print(f'  Has disturbance (>=0): {n_with_disturbance}')
if n_with_disturbance > 0:
    valid_years = year_dist[year_dist >= 0]
    print(f'  Year range: {int(np.min(valid_years))} to {int(np.max(valid_years))}')
    print(f'  Mean year: {np.mean(valid_years):.1f}')

# 4. Feature array shapes
print(f'\nFeature Array Shapes:')
print(f'  cube_idx: {training_dataset["cube_idx"].shape}')
print(f'  cube_name: {training_dataset["cube_name"].shape}')
print(f'  x: {training_dataset["x"].shape}')
print(f'  y: {training_dataset["y"].shape}')
print(f'  disturbances: {training_dataset["disturbances"].shape}')
print(f'  s2_bands: {training_dataset["s2_bands"].shape}')
print(f'  dem: {training_dataset["dem"].shape}')

# 5. Data type checks
print(f'\nData Type Verification:')
print(f'  cube_idx: {training_dataset["cube_idx"].dtype}')
print(f'  disturbances: {training_dataset["disturbances"].dtype}')
print(f'  s2_bands: {training_dataset["s2_bands"].dtype}')
print(f'  dem: {training_dataset["dem"].dtype}')

# 6. NaN/Missing value checks (dataset-wide -- see the per-year table below for why this
# alone is not enough to trust)
print(f'\nMissing Value Checks:')
print(f'  disturbances NaNs: {np.isnan(training_dataset["disturbances"].values).sum()}')
print(f'  s2_bands NaNs: {np.isnan(training_dataset["s2_bands"].values).sum()}')
print(f'  dem NaNs: {np.isnan(training_dataset["dem"].values).sum()}')

# 7. Per-year quality report, now covering the indices computed in this notebook
per_year_core_vars = ['s2_bands', 'dem', 'ndvi', 'ndwi']
if 'nbr' in training_dataset.data_vars:
    per_year_core_vars.append('nbr')

print(f'\nPer-Year Quality Report:')
display(per_year_quality_report(training_dataset, core_vars=per_year_core_vars))

# 8. Sample data
print(f'\nSample of first 3 pixels:')
for i in range(min(3, len(training_dataset.pixel))):
    pixel_data = training_dataset.isel(pixel=i)
    print(f'  Pixel {i}: cube={pixel_data["cube_name"].values}, x={pixel_data["x"].values}, y={pixel_data["y"].values}, '
          f'disturbed_years={np.sum(pixel_data["disturbances"].values == 1)}, first_disturbance_year={pixel_data["year_disturbance"].values}')

# 9. Memory usage
print(f'\nStorage Information:')
print(f'  Memory usage: {training_dataset.nbytes / 1e6:.2f} MB')

Loading saved dataset from C:\Users\bartu\Desktop\Fonda-scikit - Git\training_data_enriched.zarr...


✓ Dataset loaded successfully

=== DATASET QUALITY CHECKS ===

Dataset Overview:
  Total pixels: 8155205
  Total years: 7
  Total observations: 57086435



Disturbance Label Distribution:
  Disturbed (1): 1,132,683 (2.0%)
  Not disturbed (0): 55,945,933 (98.0%)

Year of First Disturbance:
  Never disturbed (-1): 7060710
  Has disturbance (>=0): 1094495
  Year range: 0 to 6
  Mean year: 3.2

Feature Array Shapes:
  cube_idx: (8155205,)
  cube_name: (8155205,)
  x: (8155205,)
  y: (8155205,)
  disturbances: (8155205, 7)
  s2_bands: (8155205, 7, 7)
  dem: (8155205, 7)

Data Type Verification:
  cube_idx: int32
  disturbances: uint8
  s2_bands: float64
  dem: float64

Missing Value Checks:


  disturbances NaNs: 0


  s2_bands NaNs: 57434909


  dem NaNs: 19257

Per-Year Quality Report:


,year,label_pos,label_neg,label_nodata,pos_rate_pct,s2_bands_nan_pct,dem_nan_pct,ndvi_nan_pct,ndwi_nan_pct,nbr_nan_pct,row_survival_pct
0,2016,115361,8038727,1117,1.41,100.00,0.03,100.00,100.00,100.00,0.00
1,2017,150189,8003899,1117,1.84,0.12,0.03,0.12,0.12,0.12,99.84
2,2018,171721,7982367,1117,2.11,0.11,0.03,0.11,0.11,0.11,99.84
3,2019,163167,7990921,1117,2.00,0.12,0.03,0.12,0.12,0.12,99.84
4,2020,157064,7997024,1117,1.93,0.10,0.03,0.10,0.10,0.10,99.85
5,2021,126634,8027454,1117,1.55,0.11,0.03,0.11,0.11,0.11,99.84
6,2022,248547,7905541,1117,3.05,0.05,0.03,0.05,0.05,0.05,99.91



Sample of first 3 pixels:
  Pixel 0: cube=mc_-0.04_51.09_1.2_20230627_0, x=28, y=0, disturbed_years=0, first_disturbance_year=-1


  Pixel 1: cube=mc_-0.04_51.09_1.2_20230627_0, x=30, y=0, disturbed_years=0, first_disturbance_year=-1
  Pixel 2: cube=mc_-0.04_51.09_1.2_20230627_0, x=33, y=0, disturbed_years=0, first_disturbance_year=-1

Storage Information:
  Memory usage: 5341.66 MB


In [15]:
# SOLUTION 2: Use NetCDF format instead (more Windows-friendly)
# output_path = PROJECT_ROOT / 'training_data_enriched.nc'
# print(f'Saving to NetCDF format: {output_path}...')
# ds.to_netcdf(output_path, engine='netcdf4')
# print('✓ Saved as NetCDF!')

# # To load later:
# # ds = xr.open_dataset('training_data_enriched.nc', chunks={'pixel': 10000})

# SOLUTION 3: Force Zarr v2 format (more stable on Windows)
# output_path = PROJECT_ROOT / 'training_data_enriched_v2.zarr'
# if output_path.exists():
#     shutil.rmtree(output_path)
# ds_chunked = ds.chunk({'pixel': 10000, 'year': -1})
# ds_chunked.to_zarr(output_path, mode='w', compute=True, 
#                     safe_chunks=False, zarr_version=2)
# print('✓ Saved with Zarr v2!')

# SOLUTION 4: Reduce Zarr async concurrency
# import zarr
# zarr.config.set({'async.concurrency': 1})
# output_path = PROJECT_ROOT / 'training_data_enriched.zarr'
# if output_path.exists():
#     shutil.rmtree(output_path)
#     time.sleep(1)
# with dask.config.set(scheduler='synchronous'):
#     ds_chunked = ds.chunk({'pixel': 10000, 'year': -1})
#     ds_chunked.to_zarr(output_path, mode='w', compute=True, safe_chunks=False)
# print('✓ Saved with reduced concurrency!')

# SOLUTION 5: Check for locking processes (diagnostic)
# import subprocess
# result = subprocess.run(['powershell', 'Get-Process', '|', 'Where-Object', 
#                          '{$_.Path -like "*python*"}'], 
#                         capture_output=True, text=True)
# print("Python processes running:")
# print(result.stdout)

print("Alternative solutions ready - uncomment one to try")

Alternative solutions ready - uncomment one to try


### Alternative Solutions if Above Fails

If the temp directory approach still fails, try these alternatives (uncomment and run one at a time):

## 6. Temporal Features (Year-over-Year Deltas)

In [16]:
# Compute temporal deltas for spectral indices
ds_path = PROJECT_ROOT / 'training_data_enriched.zarr'
print('Loading enriched dataset with chunking...')
ds = xr.open_dataset(ds_path, engine='zarr', chunks={'pixel': 10000})

print('Computing temporal features with special handling for year 2016...')
print('Strategy: Set 2016 to zero baseline; 2017 delta = 2017 - 0; later years = current - previous')

# For each spectral index, compute year-over-year differences
temporal_vars = ['ndvi', 'ndwi']
if 'nbr' in ds.data_vars:
    temporal_vars.append('nbr')

for var_name in temporal_vars:
    var = ds[var_name]  # dims: pixel, year
    first_year = var.year.values[0]

    # Fill NaNs with 0 and force the first year (2016) to 0
    var_filled = var.fillna(0)
    var_filled.loc[dict(year=first_year)] = 0

    # Year-over-year difference keeps the 'year' dimension starting from the second year
    delta = var_filled.diff(dim='year')

    # Reindex back to full year coords; set 2016 delta to 0 explicitly
    delta_full = delta.reindex(year=var.year, fill_value=0)

    ds[f'{var_name}_delta'] = delta_full
    print(f'✓ Added {var_name}_delta with year coords {list(delta_full.year.values)}')

# Compute years_since_last_disturbance (no data leakage)
print('\nComputing years_since_last_disturbance feature...')
dist = ds['disturbances']  # dims: pixel, year
n_years = len(ds.year)
n_pixels = len(ds.pixel)

# Initialize with sentinel value 7 (never disturbed previously)
years_since = np.full((n_pixels, n_years), 7, dtype=np.float32)

# For each year t, look only at years < t to count years since last disturbance
for t in range(n_years):
    if t == 0:
        # Year 0: no previous data, always 7
        years_since[:, t] = 7
    else:
        # Get disturbances from all years before t
        past_disturbances = dist.isel(year=slice(0, t)).values  # (n_pixels, t)
        
        # For each pixel, find the most recent disturbance year
        for p in range(n_pixels):
            # Find years where pixel was disturbed (in 0..t-1)
            disturbed_years = np.where(past_disturbances[p, :] == 1)[0]
            
            if len(disturbed_years) > 0:
                # Most recent disturbance year index
                last_disturb_idx = disturbed_years[-1]
                # Years since = current year index - last disturbance year index
                years_since[p, t] = t - last_disturb_idx
            else:
                # Never disturbed before this year
                years_since[p, t] = 7

ds['years_since_last_disturbance'] = (['pixel', 'year'], years_since)
print('✓ Added years_since_last_disturbance')

# Compute log transform
print('Computing log_years_since_last_disturbance feature...')
log_years_since = np.log1p(years_since)  # log(1 + x) to handle zeros gracefully
ds['log_years_since_last_disturbance'] = (['pixel', 'year'], log_years_since)
print('✓ Added log_years_since_last_disturbance')

# Compute ever_disturbed (binary, using only prior years)
print('Computing ever_disturbed feature...')
ever_disturbed = np.zeros((n_pixels, n_years), dtype=np.float32)

for t in range(n_years):
    if t == 0:
        # Year 0: no previous data, always 0
        ever_disturbed[:, t] = 0
    else:
        # Check if any disturbance occurred in years < t
        past_disturbances = dist.isel(year=slice(0, t)).values  # (n_pixels, t)
        # If any year before t has disturbance=1, set to 1
        ever_disturbed[:, t] = (past_disturbances.sum(axis=1) > 0).astype(np.float32)

ds['ever_disturbed'] = (['pixel', 'year'], ever_disturbed)
print('✓ Added ever_disturbed')

print(f'\nDataset now contains: {list(ds.data_vars)}')
print('Note: 2016 delta set to 0; 2017 delta = value_2017 - 0; subsequent deltas = current - previous')
print('New features use only prior years (no data leakage):')

Loading enriched dataset with chunking...


C:\Users\bartu\AppData\Local\Temp\ipykernel_1116\4241229982.py:4: UserWarning: The specified chunks separate the stored chunks along dimension "pixel" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(ds_path, engine='zarr', chunks={'pixel': 10000})


Computing temporal features with special handling for year 2016...
Strategy: Set 2016 to zero baseline; 2017 delta = 2017 - 0; later years = current - previous


✓ Added ndvi_delta with year coords [2016, 2017, 2018, 2019, 2020, 2021, 2022]
✓ Added ndwi_delta with year coords [2016, 2017, 2018, 2019, 2020, 2021, 2022]


✓ Added nbr_delta with year coords [2016, 2017, 2018, 2019, 2020, 2021, 2022]

Computing years_since_last_disturbance feature...


✓ Added years_since_last_disturbance
Computing log_years_since_last_disturbance feature...


✓ Added log_years_since_last_disturbance
Computing ever_disturbed feature...


✓ Added ever_disturbed

Dataset now contains: ['cube_idx', 'cube_name', 'dem', 'disturbances', 'nbr', 'ndvi', 'ndwi', 's2_bands', 'x', 'y', 'year_disturbance', 'ndvi_delta', 'ndwi_delta', 'nbr_delta', 'years_since_last_disturbance', 'log_years_since_last_disturbance', 'ever_disturbed']
Note: 2016 delta set to 0; 2017 delta = value_2017 - 0; subsequent deltas = current - previous
New features use only prior years (no data leakage):


In [17]:
# Save dataset with temporal features using chunked writing
import shutil
import time
import gc
import dask
import tempfile

# Close any existing datasets that might have handles open
try:
    ds.close()
except:
    pass

# Force garbage collection to release file handles
gc.collect()
time.sleep(1)

# Save to workspace directory (same approach as enriched dataset)
output_path = PROJECT_ROOT / 'training_data_with_features.zarr'

print(f'Saving to workspace directory: {output_path.absolute()}')

# Force remove existing directory if it exists
if output_path.exists():
    print(f'Removing existing {output_path}...')
    max_attempts = 3
    for attempt in range(max_attempts):
        try:
            shutil.rmtree(output_path)
            time.sleep(1)
            print(f'✓ Removed successfully')
            break
        except Exception as e:
            if attempt < max_attempts - 1:
                print(f'Attempt {attempt + 1} failed, retrying...')
                time.sleep(2)
            else:
                print(f'⚠ Warning: {e}')

# Pre-process strings to avoid "UnstableSpecificationWarning"
for var in ds.variables:
    if ds[var].dtype.kind in {'U', 'S'}:  # If it's a Unicode or Byte string
        ds[var] = ds[var].astype(object)

print(f'Saving to {output_path}...')

# Use synchronous scheduler to avoid Windows file access issues
with dask.config.set(scheduler='synchronous'):
    ds_chunked = ds.chunk({'pixel': 100000, 'year': -1})
    # Use encoding={} to let xarray create new chunk sizes instead of reusing old ones
    ds_chunked.to_zarr(
        output_path, 
        mode='w', 
        compute=True, 
        encoding={var: {'chunks': None} for var in ds_chunked.data_vars}
    )

print('✓ Saved successfully!')
print(f'File location: {output_path}')
print(f'Dataset shape: {len(ds.pixel)} pixels × {len(ds.year)} years')

Saving to workspace directory: C:\Users\bartu\Desktop\Fonda-scikit - Git\training_data_with_features.zarr
Removing existing C:\Users\bartu\Desktop\Fonda-scikit - Git\training_data_with_features.zarr...


✓ Removed successfully
Saving to C:\Users\bartu\Desktop\Fonda-scikit - Git\training_data_with_features.zarr...


C:\Users\bartu\AppData\Local\Temp\ipykernel_1116\1690592968.py:51: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds_chunked.to_zarr(


C:\Users\bartu\Desktop\Fonda-scikit - Git\venv\Lib\site-packages\zarr\api\asynchronous.py:244: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


✓ Saved successfully!
File location: C:\Users\bartu\Desktop\Fonda-scikit - Git\training_data_with_features.zarr
Dataset shape: 8155205 pixels × 7 years


## 7. Data Summary & Statistics

In [18]:
# DIAGNOSTIC: Check what's in the saved zarr file
ds_path = PROJECT_ROOT / 'training_data_with_features.zarr'

print(f'Loading from: {ds_path}')
print(f'File exists: {ds_path.exists()}')

if ds_path.exists():
    ds_check = xr.open_dataset(ds_path, engine='zarr')
    print(f'\nDataset info:')
    print(f'  Pixels: {len(ds_check.pixel)}')
    print(f'  Years: {len(ds_check.year)}')
    print(f'  Variables: {list(ds_check.data_vars.keys())}')

    # Check disturbances in detail
    dist = ds_check['disturbances'].values
    print(f'\nDisturbances array:')
    print(f'  Shape: {dist.shape}')
    print(f'  Unique values: {np.unique(dist)}')
    print(f'  Count of 1s: {np.sum(dist == 1)}')
    print(f'  Count of 0s: {np.sum(dist == 0)}')
    print(f'  Total: {dist.size}')

    # Check temporal deltas - THIS IS THE KEY TEST
    print('\n' + '='*80)
    print('TEMPORAL DELTA VERIFICATION')
    print('='*80)

    if 'ndvi_delta' in ds_check.data_vars:
        print(f'\nDelta variables found: ndvi_delta, ndwi_delta' + (', nbr_delta' if 'nbr_delta' in ds_check else ''))
        print(f'Delta shape: {ds_check["ndvi_delta"].shape}')
        print(f'Expected: {len(ds_check.pixel)} pixels x {len(ds_check.year)} years '
              f'(reindexed to the full year range; index 0 is the forced-zero 2016 baseline, not a real delta)')

        # Check NaN statistics per year for deltas
        print(f'\nNaN statistics for temporal deltas:')
        for year_idx in range(len(ds_check.year)):
            year_val = ds_check.year.values[year_idx]

            ndvi_delta = ds_check['ndvi_delta'].isel(year=year_idx).values
            ndwi_delta = ds_check['ndwi_delta'].isel(year=year_idx).values

            ndvi_delta_nan = np.isnan(ndvi_delta).sum()
            ndwi_delta_nan = np.isnan(ndwi_delta).sum()
            total_pixels = len(ds_check.pixel)

            print(f'  Year {year_val} (index {year_idx}):')
            print(f'    NDVI_delta: {ndvi_delta_nan}/{total_pixels} ({100*ndvi_delta_nan/total_pixels:.1f}%) NaN')
            print(f'    NDWI_delta: {ndwi_delta_nan}/{total_pixels} ({100*ndwi_delta_nan/total_pixels:.1f}%) NaN')

            if 'nbr_delta' in ds_check.data_vars:
                nbr_delta = ds_check['nbr_delta'].isel(year=year_idx).values
                nbr_delta_nan = np.isnan(nbr_delta).sum()
                print(f'    NBR_delta: {nbr_delta_nan}/{total_pixels} ({100*nbr_delta_nan/total_pixels:.1f}%) NaN')

        # Check that deltas match value_t - value_(t-1), both NaN-filled to 0, with year 0
        # (2016) forced to 0 regardless of its own value -- exactly what the "Compute
        # Spectral Indices" cell above computes before saving.
        print(f'\n--- Verifying delta computation accuracy ---')
        sample_pixels = np.random.choice(len(ds_check.pixel), size=min(100, len(ds_check.pixel)), replace=False)

        def filled_ndvi(year_idx):
            values = ds_check['ndvi'].isel(pixel=sample_pixels, year=year_idx).values
            values = np.where(np.isnan(values), 0, values)
            if year_idx == 0:
                values = np.zeros_like(values)  # forced baseline, independent of the actual 2016 value
            return values

        for year_idx in range(1, min(4, len(ds_check.year))):  # a few real deltas; index 0 has none to check
            year_val = ds_check.year.values[year_idx]

            ndvi_delta_expected = filled_ndvi(year_idx) - filled_ndvi(year_idx - 1)
            ndvi_delta_saved = ds_check['ndvi_delta'].isel(pixel=sample_pixels, year=year_idx).values

            matches = np.isclose(ndvi_delta_expected, ndvi_delta_saved, equal_nan=True)
            match_rate = matches.sum() / len(matches)

            print(f'  Delta at year {year_val} (index {year_idx}): {match_rate*100:.1f}% match with manual computation')
            if match_rate < 1.0:
                print(f'    WARNING: Deltas may be incorrectly computed!')
    else:
        print('⚠️ No temporal delta variables found!')

    print('='*80)
else:
    print('⚠️ File does not exist!')

Loading from: C:\Users\bartu\Desktop\Fonda-scikit - Git\training_data_with_features.zarr
File exists: True



Dataset info:
  Pixels: 8155205
  Years: 7
  Variables: ['cube_idx', 'cube_name', 'dem', 'disturbances', 'ever_disturbed', 'log_years_since_last_disturbance', 'nbr', 'nbr_delta', 'ndvi', 'ndvi_delta', 'ndwi', 'ndwi_delta', 's2_bands', 'x', 'y', 'year_disturbance', 'years_since_last_disturbance']



Disturbances array:
  Shape: (8155205, 7)


  Unique values: [  0   1 255]
  Count of 1s: 1132683
  Count of 0s: 55945933
  Total: 57086435

TEMPORAL DELTA VERIFICATION

Delta variables found: ndvi_delta, ndwi_delta, nbr_delta
Delta shape: (8155205, 7)
Expected: 8155205 pixels x 7 years (reindexed to the full year range; index 0 is the forced-zero 2016 baseline, not a real delta)

NaN statistics for temporal deltas:


  Year 2016 (index 0):
    NDVI_delta: 0/8155205 (0.0%) NaN
    NDWI_delta: 0/8155205 (0.0%) NaN


...

  Year 2022 (index 6):
    NDVI_delta: 0/8155205 (0.0%) NaN
    NDWI_delta: 0/8155205 (0.0%) NaN


    NBR_delta: 0/8155205 (0.0%) NaN

--- Verifying delta computation accuracy ---


  Delta at year 2017 (index 1): 100.0% match with manual computation


  Delta at year 2018 (index 2): 100.0% match with manual computation


  Delta at year 2019 (index 3): 100.0% match with manual computation


In [19]:
# Load final dataset and show statistics
ds_path = PROJECT_ROOT / 'training_data_with_features.zarr'
ds = xr.open_dataset(ds_path, engine='zarr')

print('\n=== TRAINING DATA SUMMARY ===')
print(f'Total pixels: {len(ds.pixel)}')
print(f'Total years: {len(ds.year)}')
print(f'Total observations: {len(ds.pixel) * len(ds.year)}')

# Show disturbance statistics
disturbances = ds['disturbances'].values.flatten()
n_disturbed = int(np.sum(disturbances == 1))
n_not_disturbed = int(np.sum(disturbances == 0))
total = n_disturbed + n_not_disturbed
print(f'\nDisturbance labels:')
print(f'  Disturbed (1): {n_disturbed} ({100.0*n_disturbed/total:.1f}%)')
print(f'  Not disturbed (0): {n_not_disturbed} ({100.0*n_not_disturbed/total:.1f}%)')

# The per-year quality report above (Section 5) already covers s2_bands/dem/ndvi/ndwi/nbr on
# this exact data -- nothing between that save and this one touches those variables again, so
# recomputing it here would just repeat a multi-GB scan for the same numbers. This section
# only adds what's new since then: shapes/dtypes of the delta and temporal features, read from
# metadata rather than `.values` so it doesn't materialize any of them.
print(f'\nFeatures available:')
for var in ds.data_vars:
    if var not in ['disturbances', 'cube_idx', 'cube_name', 'x', 'y', 'year_disturbance']:
        print(f'  {var}: shape={ds[var].shape}, dtype={ds[var].dtype}')


=== TRAINING DATA SUMMARY ===
Total pixels: 8155205
Total years: 7
Total observations: 57086435



Disturbance labels:
  Disturbed (1): 1132683 (2.0%)
  Not disturbed (0): 55945933 (98.0%)

Features available:
  dem: shape=(8155205, 7), dtype=float64
  ever_disturbed: shape=(8155205, 7), dtype=float32
  log_years_since_last_disturbance: shape=(8155205, 7), dtype=float32
  nbr: shape=(8155205, 7), dtype=float64
  nbr_delta: shape=(8155205, 7), dtype=float64
  ndvi: shape=(8155205, 7), dtype=float64
  ndvi_delta: shape=(8155205, 7), dtype=float64
  ndwi: shape=(8155205, 7), dtype=float64
  ndwi_delta: shape=(8155205, 7), dtype=float64
  s2_bands: shape=(8155205, 7, 7), dtype=float64
  years_since_last_disturbance: shape=(8155205, 7), dtype=float32


In [20]:
# Show sample of data
print('\nSample of first 5 pixels:')
sample_pixels = ds.isel(pixel=slice(0, 5))

# Create a summary dataframe
summary_data = []
for i in range(min(5, len(ds.pixel))):
    summary_data.append({
        'pixel_idx': i,
        'cube': ds['cube_name'].isel(pixel=i).values.item(),
        'x': ds['x'].isel(pixel=i).values.item(),
        'y': ds['y'].isel(pixel=i).values.item(),
        'year_first_disturb': ds['year_disturbance'].isel(pixel=i).values.item(),
        'n_disturbed_years': np.sum(ds['disturbances'].isel(pixel=i).values == 1),
    })

summary_df = pd.DataFrame(summary_data)
display(summary_df)


Sample of first 5 pixels:


,pixel_idx,cube,x,y,year_first_disturb,n_disturbed_years
0,0,mc_-0.04_51.09_1.2_20230627_0,28,0,-1,0
1,1,mc_-0.04_51.09_1.2_20230627_0,30,0,-1,0
2,2,mc_-0.04_51.09_1.2_20230627_0,33,0,-1,0
3,3,mc_-0.04_51.09_1.2_20230627_0,34,0,-1,0
4,4,mc_-0.04_51.09_1.2_20230627_0,37,0,-1,0


## 8. Create Cube-wise Train/Validation/Test Split

To avoid spatial leakage, we split by cubes (not by pixels). All pixels from the same cube will belong to the same set.

In [21]:
# Create cube-wise split to avoid spatial leakage
from sklearn.model_selection import train_test_split

ds_path = PROJECT_ROOT / 'training_data_with_features.zarr'
ds = xr.open_dataset(ds_path, engine='zarr')

print('Creating cube-wise train/validation/test split...')

# Get unique cubes
unique_cubes = np.unique(ds['cube_idx'].values)
n_cubes = len(unique_cubes)
print(f'Total unique cubes: {n_cubes}')

# Split ratios: 70% train, 15% validation, 15% test
train_ratio = 0.70
val_ratio = 0.15
test_ratio = 0.15

# First split: separate test set (15%)
cubes_train_val, cubes_test = train_test_split(
    unique_cubes,
    test_size=test_ratio,
    random_state=42
)

# Second split: separate validation from training (15% of remaining 85%)
val_ratio_adjusted = val_ratio / (train_ratio + val_ratio)
cubes_train, cubes_val = train_test_split(
    cubes_train_val,
    test_size=val_ratio_adjusted,
    random_state=42
)

print(f'\nCube split:')
print(f'  Train cubes: {len(cubes_train)} ({100*len(cubes_train)/n_cubes:.1f}%)')
print(f'  Validation cubes: {len(cubes_val)} ({100*len(cubes_val)/n_cubes:.1f}%)')
print(f'  Test cubes: {len(cubes_test)} ({100*len(cubes_test)/n_cubes:.1f}%)')

# Create pixel-level masks based on cube membership
train_mask = np.isin(ds['cube_idx'].values, cubes_train)
val_mask = np.isin(ds['cube_idx'].values, cubes_val)
test_mask = np.isin(ds['cube_idx'].values, cubes_test)

# Get pixel indices for each set
train_pixel_indices = np.where(train_mask)[0]
val_pixel_indices = np.where(val_mask)[0]
test_pixel_indices = np.where(test_mask)[0]

print(f'\nPixel split:')
print(f'  Train pixels: {len(train_pixel_indices)} ({100*len(train_pixel_indices)/len(ds.pixel):.1f}%)')
print(f'  Validation pixels: {len(val_pixel_indices)} ({100*len(val_pixel_indices)/len(ds.pixel):.1f}%)')
print(f'  Test pixels: {len(test_pixel_indices)} ({100*len(test_pixel_indices)/len(ds.pixel):.1f}%)')

# Check class balance in each set
for set_name, mask in [('Train', train_mask), ('Validation', val_mask), ('Test', test_mask)]:
    set_disturbances = ds['disturbances'].values[mask].flatten()
    # Cast to Python int before the percentage math below: np.sum(bool_array) can come back
    # as a fixed-width numpy int (observed as int32 here), and 100 * n_neg overflows it for
    # the multi-million-pixel train split. Python ints are arbitrary precision, so this can't
    # overflow regardless of split size.
    n_pos = int(np.sum(set_disturbances == 1))
    n_neg = int(np.sum(set_disturbances == 0))
    total = n_pos + n_neg
    print(f'\n{set_name} set class distribution:')
    print(f'  Disturbed (1): {n_pos} ({100.0*n_pos/total:.1f}%)')
    print(f'  Not disturbed (0): {n_neg} ({100.0*n_neg/total:.1f}%)')

Creating cube-wise train/validation/test split...


Total unique cubes: 1975

Cube split:
  Train cubes: 1381 (69.9%)
  Validation cubes: 297 (15.0%)
  Test cubes: 297 (15.0%)



Pixel split:
  Train pixels: 5597776 (68.6%)
  Validation pixels: 1273437 (15.6%)
  Test pixels: 1283992 (15.7%)



Train set class distribution:
  Disturbed (1): 770162 (2.0%)
  Not disturbed (0): 38409188 (98.0%)

Validation set class distribution:
  Disturbed (1): 173831 (2.0%)
  Not disturbed (0): 8739521 (98.0%)

Test set class distribution:
  Disturbed (1): 188690 (2.1%)
  Not disturbed (0): 8797224 (97.9%)


In [22]:
# Save the split indices for reuse across different models
from src.cube.manifest import (
    SPLIT_HASH_KEY,
    manifest_path_for,
    pixel_identity_hash_from_dataset,
    read_manifest,
)

split_path = PROJECT_ROOT / 'data_split.npz'
sampled_dataset_path = PROJECT_ROOT / 'training_data.zarr'

# These are positions along `pixel`, so they only mean anything against the exact pixel
# population `ds` holds. Stamp that population's identity into the npz so a later mismatch
# is detectable, and cross-check it against the manifest the sampler wrote -- that catches
# a derived store having been rebuilt from a different sampling run.
print('Computing pixel identity hash...')
pixel_identity_sha256 = pixel_identity_hash_from_dataset(ds)
print(f'  {pixel_identity_sha256}')

try:
    sampled_manifest = read_manifest(sampled_dataset_path)
except FileNotFoundError:
    print(f'\n⚠ No {manifest_path_for(sampled_dataset_path).name} found.')
    print('  This dataset predates manifest support, so the sampler settings behind these')
    print('  pixels are unknown and the split cannot be cross-checked. The hash below still')
    print('  gets recorded; re-run the sampling cells above to produce a manifest.')
else:
    if sampled_manifest['pixel_identity_sha256'] != pixel_identity_sha256:
        raise ValueError(
            f"Pixel identity mismatch -- refusing to write a split that cannot be trusted.\n"
            f"  {manifest_path_for(sampled_dataset_path).name}: {sampled_manifest['pixel_identity_sha256']}\n"
            f"  {ds_path.name}: {pixel_identity_sha256}\n"
            f"The feature dataset does not hold the pixels the sampler produced, so these "
            f"indices would address the wrong rows. Re-run this notebook end to end."
        )
    print(f"✓ Matches {manifest_path_for(sampled_dataset_path).name} "
          f"(seed={sampled_manifest['config'].get('random_seed')}, "
          f"created {sampled_manifest['created_at']})")

print(f'\nSaving split indices to {split_path}...')
np.savez(
    split_path,
    train_pixel_indices=train_pixel_indices,
    val_pixel_indices=val_pixel_indices,
    test_pixel_indices=test_pixel_indices,
    train_cube_indices=cubes_train,
    val_cube_indices=cubes_val,
    test_cube_indices=cubes_test,
    **{SPLIT_HASH_KEY: np.array(pixel_identity_sha256)},
)
print('✓ Split saved successfully!')

# Verify by reloading
split_data = np.load(split_path)
print(f'\nSaved split contains:')
for key in split_data.keys():
    value = split_data[key]
    if value.ndim == 0:
        print(f'  {key}: {value.item()}')
    else:
        print(f'  {key}: {len(value)} items')

Computing pixel identity hash...


  38c8d32300cda396519c585c12120494fe85424e90135de6c3eec11f30f9eeda
✓ Matches training_data.manifest.json (seed=42, created 2026-08-03T14:16:17Z)

Saving split indices to C:\Users\bartu\Desktop\Fonda-scikit - Git\data_split.npz...
✓ Split saved successfully!

Saved split contains:


  train_pixel_indices: 5597776 items
  val_pixel_indices: 1273437 items
  test_pixel_indices: 1283992 items
  train_cube_indices: 1381 items
  val_cube_indices: 297 items
  test_cube_indices: 297 items
  pixel_identity_sha256: 38c8d32300cda396519c585c12120494fe85424e90135de6c3eec11f30f9eeda


In [23]:
# Test new features - verify correctness on random pixels
print('='*80)
print('TESTING NEW TEMPORAL FEATURES')
print('='*80)

# Sample 10 random pixels from ever_disturbed == 1 pixels
np.random.seed(42)
# Find pixels where ever_disturbed is 1 at any year
ever_disturbed_any_year = (ds['ever_disturbed'].values.sum(axis=1) > 0)
ever_disturbed_pixels = np.where(ever_disturbed_any_year)[0]
n_ever_disturbed = len(ever_disturbed_pixels)
print(f'Total pixels with ever_disturbed == 1: {n_ever_disturbed}')

# Sample 10 from ever_disturbed pixels
sample_size = min(10, n_ever_disturbed)
sample_pixel_indices = np.random.choice(ever_disturbed_pixels, size=sample_size, replace=False)

print(f'\nSampling {len(sample_pixel_indices)} random pixels from ever_disturbed pixels to verify feature computation...\n')

for idx, pixel_idx in enumerate(sample_pixel_indices):
    print(f'\n--- Pixel {idx + 1} (Index {pixel_idx}) ---')
    
    # Create a DataFrame for easy viewing
    years = ds.year.values
    data = {
        'year': years,
        'disturbance': ds['disturbances'].isel(pixel=pixel_idx).values,
        'years_since': ds['years_since_last_disturbance'].isel(pixel=pixel_idx).values,
        'log_years_since': ds['log_years_since_last_disturbance'].isel(pixel=pixel_idx).values,
        'ever_disturbed': ds['ever_disturbed'].isel(pixel=pixel_idx).values,
    }
    
    df = pd.DataFrame(data)
    
    # Manual verification for years_since_last_disturbance
    print('Verification:')
    for t in range(len(years)):
        disturb_val = df.loc[t, 'disturbance']
        years_since_val = df.loc[t, 'years_since']
        ever_disturbed_val = df.loc[t, 'ever_disturbed']
        
        # Check logic
        if t == 0:
            expected_years_since = 7
            expected_ever_disturbed = 0
        else:
            # Find last disturbance in prior years
            prior_disturbances = df.loc[:t-1, 'disturbance'].values
            disturbed_years = np.where(prior_disturbances == 1)[0]
            
            if len(disturbed_years) > 0:
                expected_years_since = t - disturbed_years[-1]
                expected_ever_disturbed = 1
            else:
                expected_years_since = 7
                expected_ever_disturbed = 0
        
        status = '✓' if (years_since_val == expected_years_since and ever_disturbed_val == expected_ever_disturbed) else '✗ MISMATCH'
        print(f'  Year {t} ({years[t]}): disturb={int(disturb_val)}, years_since={years_since_val:.0f} (expected {expected_years_since}), ever_disturbed={int(ever_disturbed_val)} (expected {expected_ever_disturbed}) {status}')
    
    print('\nFull data:')
    display(df)

print('\n' + '='*80)
print('✓ Feature verification complete!')
print('='*80)

TESTING NEW TEMPORAL FEATURES


Total pixels with ever_disturbed == 1: 874516

Sampling 10 random pixels from ever_disturbed pixels to verify feature computation...


--- Pixel 1 (Index 6774494) ---
Verification:
  Year 0 (2016): disturb=0, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 1 (2017): disturb=0, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 2 (2018): disturb=0, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 3 (2019): disturb=0, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 4 (2020): disturb=1, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 5 (2021): disturb=0, years_since=1 (expected 1), ever_disturbed=1 (expected 1) ✓
  Year 6 (2022): disturb=0, years_since=2 (expected 2), ever_disturbed=1 (expected 1) ✓

Full data:


,year,disturbance,years_since,log_years_since,ever_disturbed
0,2016,0,7.0,2.079442,0.0
1,2017,0,7.0,2.079442,0.0
2,2018,0,7.0,2.079442,0.0
3,2019,0,7.0,2.079442,0.0
4,2020,1,7.0,2.079442,0.0
5,2021,0,1.0,0.693147,1.0
6,2022,0,2.0,1.098612,1.0



--- Pixel 2 (Index 6099914) ---
Verification:
  Year 0 (2016): disturb=0, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 1 (2017): disturb=0, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 2 (2018): disturb=1, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 3 (2019): disturb=0, years_since=1 (expected 1), ever_disturbed=1 (expected 1) ✓
  Year 4 (2020): disturb=0, years_since=2 (expected 2), ever_disturbed=1 (expected 1) ✓
  Year 5 (2021): disturb=0, years_since=3 (expected 3), ever_disturbed=1 (expected 1) ✓
  Year 6 (2022): disturb=0, years_since=4 (expected 4), ever_disturbed=1 (expected 1) ✓

Full data:


,year,disturbance,years_since,log_years_since,ever_disturbed
0,2016,0,7.0,2.079442,0.0
1,2017,0,7.0,2.079442,0.0
2,2018,1,7.0,2.079442,0.0
3,2019,0,1.0,0.693147,1.0
4,2020,0,2.0,1.098612,1.0
5,2021,0,3.0,1.386294,1.0
6,2022,0,4.0,1.609438,1.0


...


--- Pixel 9 (Index 427505) ---
Verification:
  Year 0 (2016): disturb=0, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 1 (2017): disturb=0, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 2 (2018): disturb=0, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 3 (2019): disturb=0, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 4 (2020): disturb=0, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 5 (2021): disturb=1, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 6 (2022): disturb=0, years_since=1 (expected 1), ever_disturbed=1 (expected 1) ✓

Full data:


,year,disturbance,years_since,log_years_since,ever_disturbed
0,2016,0,7.0,2.079442,0.0
1,2017,0,7.0,2.079442,0.0
2,2018,0,7.0,2.079442,0.0
3,2019,0,7.0,2.079442,0.0
4,2020,0,7.0,2.079442,0.0
5,2021,1,7.0,2.079442,0.0
6,2022,0,1.0,0.693147,1.0



--- Pixel 10 (Index 1616566) ---
Verification:
  Year 0 (2016): disturb=0, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 1 (2017): disturb=0, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 2 (2018): disturb=0, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 3 (2019): disturb=0, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 4 (2020): disturb=0, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 5 (2021): disturb=1, years_since=7 (expected 7), ever_disturbed=0 (expected 0) ✓
  Year 6 (2022): disturb=0, years_since=1 (expected 1), ever_disturbed=1 (expected 1) ✓

Full data:


,year,disturbance,years_since,log_years_since,ever_disturbed
0,2016,0,7.0,2.079442,0.0
1,2017,0,7.0,2.079442,0.0
2,2018,0,7.0,2.079442,0.0
3,2019,0,7.0,2.079442,0.0
4,2020,0,7.0,2.079442,0.0
5,2021,1,7.0,2.079442,0.0
6,2022,0,1.0,0.693147,1.0



✓ Feature verification complete!
